# 02b — Vista previa de aumentos y pruebas de estrés

Ejecuta este notebook **después del 02 y antes del entrenamiento**. Solo inspecciona transformaciones en memoria: no modifica imágenes, no regenera splits y no entrena el modelo.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.config import PAIRS_DIR
from src.evaluation.augmentation_preview import preview_from_path, plot_preview

print(f'Proyecto: {PROJECT_ROOT}')
print('Este notebook no ejecuta entrenamiento.')

## Seleccionar una imagen de entrenamiento

El preview usa una imagen referenciada por `train_pairs.csv`. Validation y test no se transforman aquí ni en sus loaders.

In [ ]:
train_csv = PAIRS_DIR / 'train_pairs.csv'
if not train_csv.exists():
    raise FileNotFoundError('Falta data/pairs/train_pairs.csv. Ejecuta primero el notebook 02.')

pairs = pd.read_csv(train_csv)
if pairs.empty:
    raise ValueError('train_pairs.csv no contiene pares.')

image_path = Path(pairs.iloc[0]['image_a'])
if not image_path.is_absolute():
    image_path = PROJECT_ROOT / image_path
print(f'Imagen de muestra: {image_path}')

## Preview reproducible

La semilla fija permite repetir la misma vista. `random train augmentation` representa el pipeline aleatorio de entrenamiento; el resto son escenarios deterministas y separados del test limpio.

In [ ]:
preview = preview_from_path(image_path, seed=2026)
figure = plot_preview(preview, columns=4, figsize=(14, 10))
figure

## Interpretación

- Los cambios deben dificultar moderadamente la imagen sin borrar la identidad.
- Los lentes y la sombra de barba son simulaciones sintéticas simples; **no sustituyen capturas reales**.
- Si una transformación oculta rasgos esenciales con demasiada frecuencia, ajusta su magnitud o probabilidad antes de entrenar.
- En la siguiente sesión conviene comparar test limpio, estrés sintético y una segunda sesión real por separado.